# Transcripción de manuscritos parroquiales — Gemini 2.5 Flash (Google)

Adaptación del pipeline de OpenAI para usar la API de Google Gemini (`gemini-2.5-flash`). La estructura del flujo es idéntica: transcripción de la imagen, segmentación en actas individuales y extracción estructurada de campos demográficos, con almacenamiento en SQLite.

La diferencia principal respecto al notebook de OpenAI es el uso de esquemas Pydantic para forzar salidas JSON tipadas, lo que evita errores de parseo por campos inventados o JSON mal formado.

> **Requisitos:** API key de Google Gemini · Google Drive con las imágenes del corpus · ver `REPRODUCIR.txt`.

## Instalación de dependencias

In [ ]:
!pip install google-genai pandas openpyxl pydantic --quiet

## Montaje de Google Drive

Monta Google Drive para acceder a los archivos del proyecto.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Configuración

Valores de configuración básicos para reproducibilidad. No cambia la lógica del notebook.

In [ ]:
import os
DRIVE_MOUNT = '/content/drive'
if 'DB_PATH' not in globals():
    DB_PATH = os.path.join(DRIVE_MOUNT, 'MyDrive', 'prueba_tfm', 'bautismos_losar_vera.db')
print('DB_PATH =', DB_PATH)

## Configuración de la API

In [ ]:
import getpass
from google import genai
from google.genai import types

my_api_key = getpass.getpass('Introduce tu clave de API de Google Gemini: ')
client = genai.Client(api_key=my_api_key)

# Modelo utilizado: gemini-2.5-flash, con soporte de visión y salida JSON estructurada.
GEMINI_MODEL = 'gemini-2.5-flash'

try:
    test = client.models.generate_content(model=GEMINI_MODEL, contents='Di solo: OK')
    print('Conexion con Gemini verificada:', test.text.strip())
except Exception as e:
    print('Error de conexion:', e)

## Base de datos SQLite

Crea la tabla `bautismos` si no existe. La estructura es idéntica a la del notebook de OpenAI para garantizar la compatibilidad entre ambas extracciones.

In [ ]:
import sqlite3
import pandas as pd
import os
from datetime import datetime

DB_PATH = 'bautismos_losar_vera.db'

def crear_base_de_datos():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS bautismos (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            archivo_imagen           TEXT,
            indice_acta_en_imagen    INTEGER,
            transcripcion_acta       TEXT,
            nombre_bautizado         TEXT,
            fecha_bautizo            TEXT,
            fecha_nacimiento         TEXT,
            lugar_nacimiento_bautizado TEXT,
            nombre_padre             TEXT,
            lugar_nacimiento_padre   TEXT,
            nombre_madre             TEXT,
            lugar_nacimiento_madre   TEXT,
            abuelo_paterno           TEXT,
            lugar_nacimiento_abuelo_paterno TEXT,
            abuela_paterna           TEXT,
            lugar_nacimiento_abuela_paterna TEXT,
            abuelo_materno           TEXT,
            lugar_nacimiento_abuelo_materno TEXT,
            abuela_materna           TEXT,
            lugar_nacimiento_abuela_materna TEXT,
            notas_adicionales        TEXT,
            fecha_procesado          TEXT
        )
    ''')
    conn.commit()
    conn.close()
    print(f'Base de datos creada/verificada: {DB_PATH}')

crear_base_de_datos()

## Esquemas Pydantic

Define los modelos de datos que Gemini usa como `response_schema`. Obligan al modelo a devolver JSON con exactamente los campos esperados, evitando truncamientos o campos inventados.

In [ ]:
import base64
import json
import re
from typing import Optional, List
from pydantic import BaseModel

# Esquemas Pydantic: obligan a Gemini a devolver JSON con exactamente estos
# campos (equivalente a json_object de OpenAI, pero ademas tipado). Esto evita que
# el parseo falle por campos inventados, JSON mal formado o truncamientos.

class ActaExtraida(BaseModel):
    nombre_bautizado: Optional[str] = None
    fecha_bautizo: Optional[str] = None
    fecha_nacimiento: Optional[str] = None
    lugar_nacimiento_bautizado: Optional[str] = None
    nombre_padre: Optional[str] = None
    lugar_nacimiento_padre: Optional[str] = None
    nombre_madre: Optional[str] = None
    lugar_nacimiento_madre: Optional[str] = None
    abuelo_paterno: Optional[str] = None
    lugar_nacimiento_abuelo_paterno: Optional[str] = None
    abuela_paterna: Optional[str] = None
    lugar_nacimiento_abuela_paterna: Optional[str] = None
    abuelo_materno: Optional[str] = None
    lugar_nacimiento_abuelo_materno: Optional[str] = None
    abuela_materna: Optional[str] = None
    lugar_nacimiento_abuela_materna: Optional[str] = None
    notas_adicionales: Optional[str] = None

class ListaActas(BaseModel):
    actas: List[str]


## Prompts y funciones principales

Define los tres prompts del sistema y las funciones `transcribir_imagen`, `extraer_datos`, `segmentar_actas` y `guardar_en_bd`. El parámetro `thinking_budget=0` desactiva el razonamiento interno de Gemini, que consumiría tokens invisibles y podría truncar la salida en transcripciones largas.

In [ ]:
SYSTEM_TRANSCRIPCION = """Eres un experto paleografo y archivista especializado en registros
parroquiales en castellano de Losar de la Vera (Extremadura) del siglo XIX.
Todas las imagenes son paginas DOBLES (a doble folio) del libro de bautismos de la parroquia
de Santiago Apostol de Losar de la Vera, periodo 1807 a 1825, ordenadas cronologicamente.
Cada imagen contiene aproximadamente 6-8 actas de bautismo completas (3-4 por pagina).

EJEMPLO de partida individual:
En la Iglesia Parroq.l del Sr Santiago Apostol de esta Villa del Losar en seis dias del mes de Abril
anio de mil ochocientos veinte y cinco Yo el infraescripto Cura P.ro de ella Bauticé
solemnem.te a Manuel Vicente que digeron haver nacido dia cinco de dho mes y anio
hijo legitimo de Vicente Rodriguez de Bartholome y Gabriela Gonzalez.
Abuelos Paternos Francisco Rodriguez y Bartholome y Francisca Escobar natural del Lugar
del vante Christo de Lezo en Vizcaya. Abuelos Maternos Antonio Berrocoso y Maria Gonzalez
todos naturales de esta Villa. Fue su Padrino Manuel Rodriguez de Manuel marido
de Alfonsa Hernandez a quienes adverti su obligacion y parentesco espiritual
que conste lo firme Thomas Sanchez de las Mattas.

INSTRUCCIONES ESTRICTAS:
- Transcribe el texto COMPLETO de TODAS las actas visibles en la imagen, de arriba a abajo,
  de izquierda a derecha (pagina izquierda antes que la derecha).
- Respeta la ortografia original incluyendo variantes arcaicas.
- Transcribe las abreviaturas exactamente como aparecen (Ygla, Shz, P.ro, dho, leg.o, etc.).
  NO las expandas ni normalices.
- Si una palabra es ilegible escribe [ilegible].
- Separa cada acta de bautismo con la linea: ===ACTA===
- Pon el separador al INICIO de cada acta nueva (excepto la primera si hay texto desde el principio).
- NO anadas comentarios, numeracion, introducciones ni ningun texto fuera del manuscrito.
- Devuelve UNICAMENTE el texto transcrito con los separadores ===ACTA===, sin nada mas."""

SYSTEM_SEGMENTACION = """Eres un archivista experto en registros parroquiales del siglo XIX.
Recibiras la transcripcion completa de una imagen de doble pagina con 6-8 actas de bautismo.
Las actas pueden estar separadas por ===ACTA=== o no.
Tu tarea es devolver un JSON con la lista de actas individuales.

Devuelve UNICAMENTE este JSON, sin texto adicional ni bloques json:
{"actas": ["texto completo del acta 1", "texto completo del acta 2", ...]}

Reglas:
- Si el separador ===ACTA=== esta presente, usalo para dividir.
- Si falta algun separador (cambio claro de partida), anade la division igualmente.
- Cada acta empieza con "En la Iglesia" o similar formula de apertura.
- Preserva el texto de cada acta tal cual, sin modificar ni corregir nada."""

SYSTEM_EXTRACCION = """Eres un experto en analisis de registros parroquiales del siglo XVIII y XIX
de Losar de la Vera (Extremadura, Espana). A partir de la transcripcion de UNA SOLA acta de
bautismo, extrae los campos indicados y devuelve UNICAMENTE un objeto JSON valido.
Sin texto adicional, sin bloques json, sin explicaciones.

Campos a extraer:
{
  "nombre_bautizado": "nombre completo del bautizado",
  "fecha_bautizo": "dd/mm/aaaa; null si no aparece",
  "fecha_nacimiento": "dd/mm/aaaa; null si no aparece",
  "lugar_nacimiento_bautizado": "localidad o Losar de la Vera si dice natural de esta villa; null si no se menciona",
  "nombre_padre": "nombre completo; null si no aparece",
  "lugar_nacimiento_padre": "localidad o Losar de la Vera si natural de esta villa; null si no se menciona",
  "nombre_madre": "nombre completo; null si no aparece",
  "lugar_nacimiento_madre": "localidad o Losar de la Vera si natural de esta villa; null si no se menciona",
  "abuelo_paterno": "nombre completo; null si no aparece",
  "lugar_nacimiento_abuelo_paterno": "localidad; Losar de la Vera si corresponde; null si no se menciona",
  "abuela_paterna": "nombre completo; null si no aparece",
  "lugar_nacimiento_abuela_paterna": "localidad; Losar de la Vera si corresponde; null si no se menciona",
  "abuelo_materno": "nombre completo; null si no aparece",
  "lugar_nacimiento_abuelo_materno": "localidad; Losar de la Vera si corresponde; null si no se menciona",
  "abuela_materna": "nombre completo; null si no aparece",
  "lugar_nacimiento_abuela_materna": "localidad; Losar de la Vera si corresponde; null si no se menciona",
  "notas_adicionales": "parroco firmante, padrinos y datos no contemplados; null si no hay"
}

REGLAS CRITICAS:
- natural/es de esta villa / de dha villa / de esta Va -> Losar de la Vera.
- Si no se menciona lugar de nacimiento para un familiar, asume Losar de la Vera.
- Localidades frecuentes del entorno: Jarandilla de la Vera, Viandar de la Vera, Jaraiz de la Vera,
  Cuacos de Yuste, Aldeanueva de la Vera, Villanueva de la Vera, Pasaron de la Vera, Valverde de la Vera.
- Las fechas estan en palabras (ej: doce de Abril de mil ochocientos diez y siete);
  normaliza SIEMPRE a dd/mm/aaaa.
- El bautizado es hijo legitimo/natural de X e Y: X=padre (va primero), Y=madre.
- Abuelos Paternos aparecen tras esa etiqueta; Abuelos Maternos tras la suya.
- El padrino/madrina NO es progenitor del bautizado; ponlos en notas_adicionales.
- NUNCA uses cadenas vacias; si no se puede determinar un campo, pon null.
- Devuelve SOLO el JSON, nada mas."""

def transcribir_imagen(ruta_imagen, anno_aprox):
    """
    Transcribe una imagen de doble pagina pasándole el año aproximado como contexto estático.
    """
    extension = os.path.splitext(ruta_imagen)[1].lower()
    mime_types = {
        '.jpg': 'image/jpeg', '.jpeg': 'image/jpeg',
        '.png': 'image/png', '.webp': 'image/webp', '.gif': 'image/gif'
    }
    media_type = mime_types.get(extension, 'image/jpeg')

    with open(ruta_imagen, 'rb') as f:
        imagen_bytes = f.read()

    # Contexto estático infalible basado en la posición del lote
    contexto = (
        f"Las actas pertenecen al libro de bautismos de Losar de la Vera. "
        f"CONTEXTO CRONOLÓGICO: Esta imagen corresponde aproximadamente al año {anno_aprox}. "
        "Usa esta referencia para interpretar fechas ambiguas. "
        "La imagen es DOBLE PAGINA con aprox. 6-8 actas de bautismo. "
        "Transcribe todas las actas separandolas con ===ACTA===."
    )

    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=[
            types.Content(parts=[
                types.Part(text=SYSTEM_TRANSCRIPCION + "\n\n" + contexto),
                types.Part(inline_data=types.Blob(mime_type=media_type, data=imagen_bytes))
            ])
        ],
        config=types.GenerateContentConfig(
            max_output_tokens=8192,
            # gemini-2.5-flash activa "thinking" por defecto, lo que resta presupuesto
            # al texto visible. Lo desactivamos: no aporta nada en una tarea de
            # transcripcion literal y evita truncar actas al final de la imagen.
            thinking_config=types.ThinkingConfig(thinking_budget=0)
        )
    )

    try:
        return response.text.strip()
    except Exception as e:
        fr = response.candidates[0].finish_reason if response.candidates else '?'
        print(f'   Aviso: transcripcion sin texto (finish_reason={fr}): {e}')
        return ''


def extraer_datos(transcripcion_acta, contexto_extra=""):
    """Extrae campos estructurados de UNA acta de bautismo individual."""

    prompt_dinamico = SYSTEM_EXTRACCION
    if contexto_extra:
        prompt_dinamico += f"\n\n{contexto_extra}\n"
        prompt_dinamico += "Utiliza este año como referencia principal si la fecha del manuscrito es ambigua o solo dice 'dicho año'."

    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=[
            types.Content(parts=[
                types.Part(text=prompt_dinamico + f"\n\nTranscripcion del acta:\n\n{transcripcion_acta}")
            ])
        ],
        config=types.GenerateContentConfig(
            max_output_tokens=2048,
            response_mime_type='application/json',
            response_schema=ActaExtraida,
            # Tarea determinista de extraccion: no necesitamos "razonamiento" interno,
            # y en gemini-2.5-flash ese razonamiento consume tokens invisibles que
            # antes dejaban el JSON visible truncado (causa probable del fallo original).
            thinking_config=types.ThinkingConfig(thinking_budget=0)
        )
    )

    finish_reason = response.candidates[0].finish_reason if response.candidates else '?'

    # 1. Camino preferente: el SDK ya valida y parsea el JSON por nosotros
    if getattr(response, 'parsed', None) is not None:
        return response.parsed.model_dump()

    # 2. Fallback: intentar parsear el texto crudo (por si 'parsed' viniera vacio)
    try:
        texto_json = response.text.strip()
    except Exception as e:
        print(f'   Sin texto en respuesta (finish_reason={finish_reason}): {e}')
        return {}

    texto_json = re.sub(r'^```[a-zA-Z]*\n?', '', texto_json)
    texto_json = re.sub(r'\n?```$', '', texto_json).strip()

    try:
        return json.loads(texto_json)
    except json.JSONDecodeError as e:
        print(f'   Error JSON (finish_reason={finish_reason}): {e} | Respuesta completa:\n{texto_json}')
        return {}

def segmentar_actas(transcripcion_completa):
    """
    Divide la transcripcion de una imagen en actas individuales.
    Si hay 4+ partes separadas por ===ACTA=== las usa directamente;
    si no, llama a Gemini para resegmentar.
    """
    partes = [p.strip() for p in transcripcion_completa.split('===ACTA===') if p.strip()]

    if len(partes) >= 4:
        return partes

    print(f'   Aviso: solo {len(partes)} segmento(s). Solicitando resegmentacion a Gemini...')
    try:
        response = client.models.generate_content(
            model=GEMINI_MODEL,
            contents=[
                types.Content(parts=[
                    types.Part(text=SYSTEM_SEGMENTACION + f"\n\nTranscripcion completa:\n\n{transcripcion_completa}")
                ])
            ],
            config=types.GenerateContentConfig(
                max_output_tokens=8192,
                response_mime_type='application/json',
                response_schema=ListaActas,
                thinking_config=types.ThinkingConfig(thinking_budget=0)
            )
        )

        if getattr(response, 'parsed', None) is not None:
            actas = [a.strip() for a in response.parsed.actas if a.strip()]
            if actas:
                return actas

        texto = response.text.strip()
        texto = re.sub(r'^```[a-zA-Z]*\n?', '', texto)
        texto = re.sub(r'\n?```$', '', texto).strip()
        datos = json.loads(texto)
        actas = [a.strip() for a in datos.get('actas', []) if a.strip()]
        if actas:
            return actas
    except Exception as e:
        print(f'   Aviso: fallo la resegmentacion con Gemini ({e}); se usan los segmentos originales.')

    return partes if partes else [transcripcion_completa]

def guardar_en_bd(archivo_imagen, indice_acta, transcripcion_acta, datos):
    """Inserta un registro de bautismo individual en SQLite. Devuelve el ID insertado."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("""
        INSERT INTO bautismos (
            archivo_imagen, indice_acta_en_imagen, transcripcion_acta,
            nombre_bautizado, fecha_bautizo, fecha_nacimiento, lugar_nacimiento_bautizado,
            nombre_padre, lugar_nacimiento_padre,
            nombre_madre, lugar_nacimiento_madre,
            abuelo_paterno, lugar_nacimiento_abuelo_paterno,
            abuela_paterna, lugar_nacimiento_abuela_paterna,
            abuelo_materno, lugar_nacimiento_abuelo_materno,
            abuela_materna, lugar_nacimiento_abuela_materna,
            notas_adicionales, fecha_procesado
        ) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
    """, (
        archivo_imagen, indice_acta, transcripcion_acta,
        datos.get('nombre_bautizado'),
        datos.get('fecha_bautizo'),
        datos.get('fecha_nacimiento'),
        datos.get('lugar_nacimiento_bautizado'),
        datos.get('nombre_padre'),
        datos.get('lugar_nacimiento_padre'),
        datos.get('nombre_madre'),
        datos.get('lugar_nacimiento_madre'),
        datos.get('abuelo_paterno'),
        datos.get('lugar_nacimiento_abuelo_paterno'),
        datos.get('abuela_paterna'),
        datos.get('lugar_nacimiento_abuela_paterna'),
        datos.get('abuelo_materno'),
        datos.get('lugar_nacimiento_abuelo_materno'),
        datos.get('abuela_materna'),
        datos.get('lugar_nacimiento_abuela_materna'),
        datos.get('notas_adicionales'),
        datetime.now().isoformat()
    ))
    conn.commit()
    nuevo_id = cursor.lastrowid
    conn.close()
    return nuevo_id

print("Funciones cargadas correctamente.")

## Configuración del corpus y montaje de Drive

Monta Google Drive y recoge las imágenes del lote indicado. Las imágenes ya procesadas en sesiones anteriores se omiten automáticamente.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import glob
import re

# ── PARÁMETROS A AJUSTAR POR LOTE ───────────────────────────────────────────
CARPETA_DRIVE = '/content/drive/MyDrive/prueba_tfm/imagenes/prueba'
ANIO_INICIO_LOTE = 1807
ANIO_FIN_LOTE    = 1825
DESDE = None   # número mínimo de imagen (inclusive); None = sin límite
HASTA = None   # número máximo de imagen (inclusive); None = sin límite
SALTAR_YA_PROCESADAS = True  # True = reanuda donde se quedó
# ────────────────────────────────────────────────────────────────────────────

def orden_natural(ruta):
    m = re.search(r'\((\d+)\)', os.path.basename(ruta))
    return int(m.group(1)) if m else 0

# Recopilar y ordenar imágenes
todas = []
for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG'):
    todas.extend(glob.glob(os.path.join(CARPETA_DRIVE, ext)))
todas = sorted(set(todas), key=orden_natural)

# Filtrar por rango si se especifica
if DESDE is not None or HASTA is not None:
    todas = [r for r in todas
             if (DESDE is None or orden_natural(r) >= DESDE)
             and (HASTA is None or orden_natural(r) <= HASTA)]

# Omitir imágenes ya procesadas en sesiones anteriores
if SALTAR_YA_PROCESADAS:
    conn = sqlite3.connect(DB_PATH)
    ya_nombres = set(
        os.path.basename(row[0])
        for row in conn.execute('SELECT DISTINCT archivo_imagen FROM bautismos').fetchall()
    )
    conn.close()
    imagenes = [r for r in todas if os.path.basename(r) not in ya_nombres]
    omitidas = len(todas) - len(imagenes)
    if omitidas:
        print(f'  {omitidas} imagenes ya en BD -> omitidas.')
else:
    imagenes = todas

total = len(imagenes)
print(f'{total} imagenes a procesar.')

## Procesamiento por lotes

Recorre las imágenes, transcribe cada una, segmenta las actas y extrae los campos demográficos. Cada acta se guarda como una fila independiente en la base de datos.

In [ ]:
transcripciones_por_imagen = {}
registros_procesados = []

for idx, ruta in enumerate(imagenes, start=1):
    if not os.path.exists(ruta):
        print(f'Archivo no encontrado: {ruta}')
        continue

    nombre_archivo = os.path.basename(ruta)

    # 1. Cálculo matemático estricto del año para esta imagen
    fraccion = idx / total
    anno_aprox = int(ANIO_INICIO_LOTE + fraccion * (ANIO_FIN_LOTE - ANIO_INICIO_LOTE))

    print(f'\n[{idx}/{total}] {nombre_archivo} -> Año estimado: ~{anno_aprox}')

    # 2. Transcripción de la imagen completa
    try:
        print('   -> Transcribiendo...')
        texto_completo = transcribir_imagen(ruta, anno_aprox=anno_aprox)
        transcripciones_por_imagen[ruta] = texto_completo
        print(f'   OK Transcripcion: {len(texto_completo)} caracteres')
    except Exception as e:
        print(f'   ERROR al transcribir: {e}')
        continue

    # 3. Segmentación en actas individuales
    actas = segmentar_actas(texto_completo)
    print(f'   OK Actas detectadas: {len(actas)}')

    # 4. Extracción y guardado de cada acta (1 acta = 1 fila en BD)
    actas_ok = 0
    for i_acta, texto_acta in enumerate(actas, start=1):
        print(f'   -> Acta {i_acta}/{len(actas)}: extrayendo...')
        try:
            # Pasamos el año aproximado para evitar alucinaciones temporales
            prompt_contexto = f"CONTEXTO: Esta acta pertenece al año {anno_aprox} aproximadamente."
            datos = extraer_datos(texto_acta, contexto_extra=prompt_contexto)

            nuevo_id = guardar_en_bd(ruta, i_acta, texto_acta, datos)

            nombre_b = datos.get('nombre_bautizado', '[sin nombre]')
            fecha_b  = datos.get('fecha_bautizo', '?')
            print(f'   OK ID {nuevo_id}: {nombre_b} ({fecha_b})')

            registros_procesados.append({'id': nuevo_id, 'archivo': nombre_archivo,
                                          'acta': i_acta, **datos})
            actas_ok += 1
        except Exception as e:
            print(f'   ERROR en acta {i_acta}: {e}')

    print(f'   -- {actas_ok}/{len(actas)} actas guardadas de esta imagen')

print(f'\nProcesamiento completado para el lote {ANIO_INICIO_LOTE}-{ANIO_FIN_LOTE}.')
print(f'   Imagenes procesadas : {len(transcripciones_por_imagen)}/{total}')
print(f'   Registros en BD     : {len(registros_procesados)}')

## Copia de seguridad de las transcripciones brutas

Guarda el texto completo por imagen en un CSV en Drive, antes de la segmentación, para trazabilidad y depuración.

In [ ]:
filas = [
    {'numero': orden_natural(ruta), 'archivo': os.path.basename(ruta), 'transcripcion': texto}
    for ruta, texto in transcripciones_por_imagen.items()
]
filas.sort(key=lambda x: x['numero'])

df_trans = pd.DataFrame(filas)
ruta_csv = '/content/drive/MyDrive/prueba_tfm/GEMINI_transcripciones_1807-1825.csv'
df_trans.to_csv(ruta_csv, sep=';', index=False, encoding='utf-8-sig')
print(f'{len(df_trans)} transcripciones guardadas en:\n   {ruta_csv}')
if filas:
    print(f'   Rango: B ({filas[0]["numero"]}).jpg -> B ({filas[-1]["numero"]}).jpg')
df_trans[['numero', 'archivo', 'transcripcion']].head(3)

## Visualización de la base de datos

Muestra todos los registros insertados en la sesión actual.

In [ ]:
conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query('SELECT * FROM bautismos ORDER BY id', conn)
conn.close()

print(f'Total de registros en la base de datos: {len(df)}')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
df

## Exportación a CSV y Excel

Genera dos archivos: un CSV plano y un Excel con una pestaña de datos y otra de resumen de cobertura por campo.

In [ ]:
conn = sqlite3.connect(DB_PATH)
df_export = pd.read_sql_query('SELECT * FROM bautismos ORDER BY id', conn)
conn.close()

csv_path = 'bautismos_1807-1825_GEMINI.csv'
df_export.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f'CSV exportado: {csv_path}')

xlsx_path = 'bautismos_1807-1825_GEMINI.xlsx'
with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
    df_export.to_excel(writer, index=False, sheet_name='Bautismos')
    resumen = pd.DataFrame({
        'Campo': ['Total registros', 'Con fecha bautizo', 'Con fecha nacimiento',
                  'Con nombre padre', 'Con nombre madre', 'Con abuelos paternos', 'Con abuelos maternos'],
        'Cantidad': [
            len(df_export),
            df_export['fecha_bautizo'].notna().sum(),
            df_export['fecha_nacimiento'].notna().sum(),
            df_export['nombre_padre'].notna().sum(),
            df_export['nombre_madre'].notna().sum(),
            (df_export['abuelo_paterno'].notna() | df_export['abuela_paterna'].notna()).sum(),
            (df_export['abuelo_materno'].notna() | df_export['abuela_materna'].notna()).sum(),
        ]
    })
    resumen['Porcentaje'] = (resumen['Cantidad'] / len(df_export) * 100).round(1).astype(str) + '%'
    resumen.to_excel(writer, index=False, sheet_name='Resumen')
print(f'Excel exportado: {xlsx_path}')